# ARC-Autoresearch Experiment Analysis

Analysis of autonomous agent experimentation results from `results.tsv`.

**Metric**: `rhae_score` — weighted-average Relative Human-Action Efficiency (higher is better).  
**TSV columns**: `commit · rhae_score · avg_actions · goal_reached_pct · status · description`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Load results (tab-separated, never comma-separated)
df = pd.read_csv("results.tsv", sep="\t")
df["rhae_score"]       = pd.to_numeric(df["rhae_score"],       errors="coerce")
df["avg_actions"]      = pd.to_numeric(df["avg_actions"],      errors="coerce")
df["goal_reached_pct"] = pd.to_numeric(df["goal_reached_pct"], errors="coerce")
df["status"]           = df["status"].str.strip().str.upper()
df["exp_idx"]          = range(len(df))

print(f"Total experiments : {len(df)}")
print(f"Columns           : {list(df.columns)}")
df.head(10)

In [ ]:
counts    = df["status"].value_counts()
n_keep    = counts.get("KEEP",    0)
n_discard = counts.get("DISCARD", 0)
n_crash   = counts.get("CRASH",   0)
n_decided = n_keep + n_discard

print("Experiment outcomes:")
print(counts.to_string())
if n_decided > 0:
    print(f"\nKeep rate : {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")
    print(f"Crash rate: {n_crash}/{len(df)} = {n_crash / len(df):.1%}")

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    print(
        f"  #{i:3d}  rhae={row['rhae_score']:.4f}  "
        f"actions={row['avg_actions']:.1f}  "
        f"goal={row['goal_reached_pct']:.1%}  "
        f"{row['description']}"
    )

## RHAE Score Over Time

The running maximum of `rhae_score` across kept experiments shows the research frontier.  
Higher is better — RHAE = 1.0 means the agent exactly matches human efficiency.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)

baseline_rhae = valid.loc[0, "rhae_score"]

# Discarded — faint background scatter
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["rhae_score"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Kept — prominent teal dots
kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["rhae_score"],
           c="#01696f", s=55, zorder=4, label="Kept",
           edgecolors="black", linewidths=0.5)

# Running maximum step line
kept_mask = valid["status"] == "KEEP"
kept_idx  = valid.index[kept_mask]
kept_rhae = valid.loc[kept_mask, "rhae_score"]
running_max = kept_rhae.cummax()
ax.step(kept_idx, running_max, where="post",
        color="#0c4e54", linewidth=2, alpha=0.8, zorder=3, label="Running best")

# Baseline horizontal reference
ax.axhline(baseline_rhae, color="#aaaaaa", linestyle="--",
           linewidth=1.2, alpha=0.7, label=f"Baseline ({baseline_rhae:.4f})")

# Annotate each kept experiment
for idx, rhae in zip(kept_idx, kept_rhae):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, rhae),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#0c4e54", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept  = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("RHAE Score (higher is better)", fontsize=12)
ax.set_title(
    f"ARC-Autoresearch Progress — {n_total} Experiments, {n_kept} Kept Improvements",
    fontsize=14
)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Action Efficiency vs. RHAE

Lower average actions + higher RHAE score = the agent learned to solve tasks more parsimoniously.  
Points in the top-left quadrant represent the Pareto-optimal experiments.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

color_map = {"KEEP": "#01696f", "DISCARD": "#cccccc", "CRASH": "#a12c7b"}
size_map  = {"KEEP": 70,         "DISCARD": 20,         "CRASH": 30}

for status, grp in df.groupby("status"):
    valid_grp = grp.dropna(subset=["avg_actions", "rhae_score"])
    ax.scatter(
        valid_grp["avg_actions"], valid_grp["rhae_score"],
        c=color_map.get(status, "#888888"),
        s=size_map.get(status, 20),
        alpha=0.75, zorder=3 if status == "KEEP" else 2,
        edgecolors="black" if status == "KEEP" else "none",
        linewidths=0.5,
        label=status.capitalize()
    )

# Label kept experiments
for _, row in df[df["status"] == "KEEP"].iterrows():
    if pd.notna(row["avg_actions"]) and pd.notna(row["rhae_score"]):
        desc = str(row["description"])[:30]
        ax.annotate(desc, (row["avg_actions"], row["rhae_score"]),
                    textcoords="offset points", xytext=(5, 4),
                    fontsize=7.5, color="#0c4e54", alpha=0.9)

ax.set_xlabel("Avg Actions per Episode", fontsize=12)
ax.set_ylabel("RHAE Score", fontsize=12)
ax.set_title("Action Efficiency vs. RHAE Score", fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("efficiency_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## Goal Completion Rate Over Time

Tracks what fraction of the 25 public tasks the agent successfully completed per experiment.  
Separates progress in goal-reaching from pure action-efficiency gains.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)

bar_colors = ["#01696f" if s == "KEEP" else "#cccccc" for s in valid["status"]]
ax.bar(valid.index, valid["goal_reached_pct"],
       color=bar_colors, alpha=0.85, width=0.7)

# Running maximum completion rate
kept_mask = valid["status"] == "KEEP"
kept_idx  = valid.index[kept_mask]
running_max = valid.loc[kept_mask, "goal_reached_pct"].cummax()
ax.step(kept_idx, running_max, where="post",
        color="#0c4e54", linewidth=2, alpha=0.8, zorder=3, label="Running best (kept)")

keep_patch    = mpatches.Patch(color="#01696f", label="Kept")
discard_patch = mpatches.Patch(color="#cccccc", label="Discarded")
ax.legend(handles=[keep_patch, discard_patch], fontsize=9, loc="lower right")

ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Goal Reached %", fontsize=12)
ax.set_title("Goal Completion Rate per Experiment", fontsize=14)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.grid(True, axis="y", alpha=0.2)

plt.tight_layout()
plt.savefig("goal_completion.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary Statistics

In [ ]:
kept          = df[df["status"] == "KEEP"].copy()
baseline_rhae = df.iloc[0]["rhae_score"]
best_rhae     = kept["rhae_score"].max()
best_row      = kept.loc[kept["rhae_score"].idxmax()]

print(f"Baseline rhae_score : {baseline_rhae:.4f}")
print(f"Best rhae_score     : {best_rhae:.4f}")
print(f"Total improvement   : +{best_rhae - baseline_rhae:.4f}  ({(best_rhae - baseline_rhae) / max(baseline_rhae, 1e-9) * 100:.2f}%)")
print(f"Best experiment     : {best_row['description']}")
print(f"Best avg_actions    : {best_row['avg_actions']:.1f}")
print(f"Best goal_reached   : {best_row['goal_reached_pct']:.1%}")
print()

print("Cumulative RHAE improvements (kept only):")
kept_sorted = kept.reset_index()
for _, row in kept_sorted.iterrows():
    desc = str(row["description"]).strip()
    print(
        f"  Exp #{row['index']:3d}  rhae={row['rhae_score']:.4f}  "
        f"actions={row['avg_actions']:.1f}  {desc}"
    )

## Top Hits — Ranked by Incremental RHAE Gain

Each kept experiment's delta is measured against the *previous* kept experiment,  
since experiments are cumulative — each builds on the last kept state.

In [ ]:
kept          = df[df["status"] == "KEEP"].copy()
kept["prev"]  = kept["rhae_score"].shift(1)
kept["delta"] = kept["rhae_score"] - kept["prev"]

# Drop baseline row (no delta)
hits = kept.iloc[1:].copy().sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'RHAE':>7}  {'Actions':>8}  Description")
print("-" * 85)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(
        f"{rank:4d}  {row['delta']:+.4f}  {row['rhae_score']:.4f}  "
        f"{row['avg_actions']:8.1f}  {row['description']}"
    )

total_gain = hits["delta"].sum()
print(f"\n{'':>4}  {total_gain:+.4f}  {'':>7}  {'':>8}  TOTAL improvement over baseline")

## RHAE Penalty Analysis

The RHAE power-law penalty `(human_baseline / actions_taken)^2` means exceeding the human  
baseline by 2× costs more than exceeding it by 1.5×.  
This cell shows how close the agent is getting to the 1.0 ceiling per experiment.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: RHAE ceiling gap per kept experiment ---
kept = df[df["status"] == "KEEP"].copy().reset_index(drop=True)
kept["ceiling_gap"] = 1.0 - kept["rhae_score"]

ax1.bar(kept.index, kept["ceiling_gap"], color="#01696f", alpha=0.8)
ax1.set_xlabel("Kept Experiment Index", fontsize=11)
ax1.set_ylabel("Distance from RHAE = 1.0", fontsize=11)
ax1.set_title("RHAE Ceiling Gap (Kept Experiments)", fontsize=13)
ax1.grid(True, axis="y", alpha=0.2)

# --- Right: theoretical RHAE vs. action ratio ---
ratios = np.linspace(0.2, 2.0, 300)   # actions / human_baseline
rhae_curve = np.where(ratios <= 1.0, 1.0, (1.0 / ratios) ** 2)
ax2.plot(ratios, rhae_curve, color="#01696f", linewidth=2.5)
ax2.axvline(1.0, color="#aaaaaa", linestyle="--", linewidth=1.2,
            label="Human baseline (ratio = 1.0)")
ax2.fill_between(ratios, rhae_curve, alpha=0.12, color="#01696f")
ax2.set_xlabel("actions_taken / human_baseline", fontsize=11)
ax2.set_ylabel("RHAE Score", fontsize=11)
ax2.set_title("RHAE Penalty Curve  (exponent = 2)", fontsize=13)
ax2.legend(fontsize=9)
ax2.set_ylim(0, 1.1)
ax2.grid(True, alpha=0.2)

# Overlay actual kept experiment ratios if avg_actions is available
# (approximate: uses avg_actions as a proxy — per-task ratios are in run logs)
if "avg_actions" in df.columns:
    # Use baseline human_baseline estimate from first kept row's actions/rhae
    first_kept = df[df["status"] == "KEEP"].iloc[0]
    if first_kept["rhae_score"] > 0:
        est_baseline = first_kept["avg_actions"] * (first_kept["rhae_score"] ** 0.5)
        for _, row in df[df["status"] == "KEEP"].iterrows():
            if pd.notna(row["avg_actions"]) and est_baseline > 0:
                ratio_est = row["avg_actions"] / est_baseline
                ax2.scatter(ratio_est, row["rhae_score"],
                            c="#0c4e54", s=50, zorder=4,
                            edgecolors="black", linewidths=0.5)

plt.tight_layout()
plt.savefig("penalty_analysis.png", dpi=150, bbox_inches="tight")
plt.show()